# Check mask coverage and tiled datasets against data_mapping

Verifies 1-1 mapping between slides in `data_mapping.csv` and:
- **MLflow masks** (downloaded from MLflow): tissue, epithelium
- **QC masks**: blur (`Piqe_piqe_median_activity_mask_`), folding (`FoldingFunction_folding_test_`), residual (`ResidualArtifactsAndCoverage_coverage_mask_`)
- **Slide files** listed in the `path` column: checks that every slide can be opened without error
- **Tiled datasets** (MLflow): checks `slides.parquet` ↔ `data_mapping` and `tiles.parquet` ↔ `slides.parquet` consistency

In [1]:
!python -m ensurepip --upgrade
!python -m pip install pandas mlflow>3 openslide-python tifffile matplotlib

Looking in links: /tmp/tmprwve6ksi

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
from mlflow.artifacts import download_artifacts
from openslide import OpenSlide
import os
from tifffile import TiffFile

In [3]:
MLFLOW_TRACKING_URI = "http://mlflow-s3.rationai-mlflow"
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

PROJECT_PATH = Path("/mnt/projects/mammaprint")
DATA_MAPPING_CSV = PROJECT_PATH / "data_mapping.csv"
QC_MASKS_PATH = PROJECT_PATH / "qc_masks3"

MLFLOW_MASKS = {
    # "tissue": "mlflow-artifacts:/3/50c597ac29784808bd89d55d9912f699/artifacts/tissue_masks",
    # "epithelium": "mlflow-artifacts:/3/c5e6c447e4784e5b991bff258195b5df/artifacts/epithelium_masks",
    "tissue": "mlflow-artifacts:/3/1d67612f123e46f6a3c6183257106512/artifacts/tissue_masks",
    "epithelium": "mlflow-artifacts:/3/0176fb9485de49699b1e16f362cff5fa/artifacts/epithelium_masks",
}

QC_MASK_PREFIXES = {
    "blur": "Piqe_piqe_median_activity_mask_",
    "folding": "FoldingFunction_folding_test_",
    "residual": "ResidualArtifactsAndCoverage_coverage_mask_",
}

TILED_DATASETS = {
    # "mou_3_224": "mlflow-artifacts:/3/0b32eceb1c434abf94104bb2c54b792e/artifacts/mou_3_224",
    # "mou_2_224": "mlflow-artifacts:/3/1ebe489fd3ca4f5682c9a4b3d6f74620/artifacts/mou_2_224",
}

EMBEDDINGS = {
    # "embedding3": "mlflow-artifacts:/3/16e8f2a772ec48d2a21dc8cf704f0e10/artifacts/embeddings",
}

In [4]:
# Load data mapping
df = pd.read_csv(DATA_MAPPING_CSV)
print(f"Loaded {len(df)} slides from {DATA_MAPPING_CSV}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 2042 slides from /mnt/projects/mammaprint/data_mapping.csv
Columns: ['record_num', 'mammaprint_index', 'type', 'path', 'split']


,record_num,mammaprint_index,type,path,split
0,2023/00835,-0.158,b luminal,/mnt/data/MOU/breast/mammaprint/P2023_0835,test
1,2023/00836,0.058,a luminal,/mnt/data/MOU/breast/mammaprint/P2023_0836,train
2,2023/00837,-0.039,b luminal,/mnt/data/MOU/breast/mammaprint/P2023_0837,train
3,2023/00838,0.065,a luminal,/mnt/data/MOU/breast/mammaprint/P2023_0838,test
4,2023/00839,-0.429,b luminal,/mnt/data/MOU/breast/mammaprint/P2023_0839,test


In [5]:
# Derive the .tiff name that the tiling pipeline expects for each slide
df["tiff_name"] = df["path"].apply(lambda p: Path(p).with_suffix(".tiff").name)
expected_names = set(df["tiff_name"])
print(f"{len(expected_names)} unique slide tiff names expected")

2042 unique slide tiff names expected


In [6]:
# ---- Verify data_mapping slide files can be opened ----
SLIDE_SUFFIX_CANDIDATES = (".mrxs", ".tiff", ".tif")


def _slide_path_candidates(raw_path: str) -> list[Path]:
    path = Path(raw_path)
    candidates = [path]
    if not path.suffix:
        candidates.extend(Path(f"{raw_path}{suffix}") for suffix in SLIDE_SUFFIX_CANDIDATES)
    return candidates


def _resolve_slide_path(raw_path: str) -> Path:
    for candidate in _slide_path_candidates(raw_path):
        if candidate.is_file():
            return candidate
    return Path(raw_path)


def _try_open_slide(raw_path: str) -> tuple[Path, str | None]:
    slide_path = _resolve_slide_path(raw_path)
    try:
        str_path = str(slide_path)
        if str_path.lower().endswith((".ome.tiff", ".ome.tif")):
            with TiffFile(str_path) as slide:
                _ = len(slide.pages)
        else:
            with OpenSlide(str_path) as slide:
                _ = slide.dimensions
        return slide_path, None
    except Exception as e:
        return slide_path, f"{type(e).__name__}: {e}"


# slide_open_rows = []
# for row in df.itertuples(index=False):
#     opened_path, error = _try_open_slide(row.path)
#     if error:
#         slide_open_rows.append(
#             {
#                 "record_num": row.record_num,
#                 "path": row.path,
#                 "resolved_path": str(opened_path),
#                 "error": error,
#             }
#         )

# slide_open_errors = pd.DataFrame(slide_open_rows)
# if slide_open_errors.empty:
#     print(f"All {len(df)} slide files opened successfully.")
# else:
#     print(f"{len(slide_open_errors)} slide files failed to open:")
#     display(slide_open_errors)
#     raise AssertionError(f"{len(slide_open_errors)} slide files failed to open")

In [7]:
# ---- MLflow masks (download from MLflow) ----
mlflow_dirs: dict[str, Path] = {}
mlflow_files: dict[str, set[str]] = {}

for mask_name, uri in MLFLOW_MASKS.items():
    mask_dir = Path(download_artifacts(uri))
    mlflow_dirs[mask_name] = mask_dir
    found = {p.name for p in mask_dir.iterdir() if p.is_file()}
    mlflow_files[mask_name] = found
    print(f"{len(found)} {mask_name} mask files found in {mask_dir}")

2042 tissue mask files found in /tmp/tmpm4swn0wz/tissue_masks


MlflowException: The following failures occurred while downloading one or more artifacts from http://mlflow-s3.rationai-mlflow/api/2.0/mlflow-artifacts/artifacts/3/0176fb9485de49699b1e16f362cff5fa/artifacts:
##### File epithelium_masks/P2023_01448.tiff #####
Response ended prematurely
##### File epithelium_masks/P2023_03021.tiff #####
Response ended prematurely
##### File epithelium_masks/P2024_02850.tiff #####
Response ended prematurely

In [ ]:
# ---- QC masks (local directory) ----
qc_files: dict[str, set[str]] = {}
for mask_type, prefix in QC_MASK_PREFIXES.items():
    found = {
        p.name.removeprefix(prefix)
        for p in QC_MASKS_PATH.iterdir()
        if p.is_file() and p.name.startswith(prefix)
    }
    qc_files[mask_type] = found
    print(f"{len(found)} {mask_type} mask files found (prefix: {prefix})")

In [ ]:
# ---- Build per-slide report ----
rows = []
for name in sorted(expected_names):
    row = {"tiff_name": name}
    for mask_name in MLFLOW_MASKS:
        row[mask_name] = name in mlflow_files[mask_name]
    for mask_type in QC_MASK_PREFIXES:
        row[mask_type] = name in qc_files[mask_type]
    rows.append(row)

mask_cols = [*MLFLOW_MASKS.keys(), *QC_MASK_PREFIXES.keys()]
report = pd.DataFrame(rows)
report["all_present"] = report[mask_cols].all(axis=1)
report

In [ ]:
# # ---- Try opening each mask file to verify readability ----
# def _try_open_mask(path: Path) -> str | None:
#     """Try to open a mask file. Returns None on success or an error message."""
#     try:
#         str_path = str(path)
#         if str_path.lower().endswith((".ome.tiff", ".ome.tif")):
#             with TiffFile(str_path) as slide:
#                 pass
#         else:
#             with OpenSlide(str_path) as slide:
#                 pass
#         return None
#     except Exception as e:
#         return f"{type(e).__name__}: {e}"


# open_errors: list[dict[str, str]] = []

# # MLflow masks
# for mask_name in MLFLOW_MASKS:
#     for name in sorted(expected_names & mlflow_files[mask_name]):
#         path = mlflow_dirs[mask_name] / name
#         err = _try_open_mask(path)
#         if err:
#             open_errors.append({"mask_type": mask_name, "tiff_name": name, "error": err})

# # QC masks
# for mask_type, prefix in QC_MASK_PREFIXES.items():
#     for name in sorted(expected_names & qc_files[mask_type]):
#         path = QC_MASKS_PATH / f"{prefix}{name}"
#         err = _try_open_mask(path)
#         if err:
#             open_errors.append({"mask_type": mask_type, "tiff_name": name, "error": err})

# if open_errors:
#     errors_df = pd.DataFrame(open_errors)
#     print(f"{len(open_errors)} mask files failed to open:")
#     display(errors_df)
# else:
#     print("All existing mask files opened successfully.")

In [ ]:
# ---- Summary ----
summary = report[mask_cols].sum().to_frame("found")
summary["missing"] = len(expected_names) - summary["found"]
summary["total"] = len(expected_names)
print("Coverage summary:")
summary

In [ ]:
# ---- Slides missing at least one mask ----
missing = report[~report["all_present"]].drop(columns=["all_present"])
print(f"{len(missing)} slides missing at least one mask:")
missing

In [ ]:
# ---- Extra masks not in data_mapping ----
for mask_name in MLFLOW_MASKS:
    extra = mlflow_files[mask_name] - expected_names
    print(f"{len(extra)} extra {mask_name} masks with no matching slide in data_mapping:")
    if extra:
        print(sorted(extra))
    print()

for mask_type in QC_MASK_PREFIXES:
    extra = qc_files[mask_type] - expected_names
    print(f"{len(extra)} extra {mask_type} masks with no matching slide:")
    if extra:
        print(sorted(extra))

## Visual inspection

Display 5 sample masks for each mask type (MLflow + QC) in a grid.

In [ ]:
# ---- Visual inspection: 5 sample masks per type ----
N_SAMPLES = 5
THUMB_SIZE = (512, 512)


def _read_mask_thumbnail(path: Path) -> np.ndarray | None:
    """Read a mask file and return a thumbnail as a numpy array."""
    try:
        str_path = str(path)
        if str_path.lower().endswith((".ome.tiff", ".ome.tif")):
            with TiffFile(str_path) as tif:
                page = tif.pages[0]
                # Read full page then downsample
                img = page.asarray()
                # Resize to thumbnail via slicing (nearest-neighbor)
                step_y = max(1, img.shape[0] // THUMB_SIZE[1])
                step_x = max(1, img.shape[1] // THUMB_SIZE[0])
                return img[::step_y, ::step_x]
        else:
            with OpenSlide(str_path) as slide:
                thumb = slide.get_thumbnail(THUMB_SIZE)
                return np.array(thumb)
    except Exception as e:
        print(f"  Could not read {path.name}: {e}")
        return None


# Collect mask paths: list of (mask_type, [paths])
all_mask_types: list[tuple[str, list[Path]]] = []

# Pick 5 sample tiff names that exist across all available mask types
sample_names = sorted(expected_names)[:N_SAMPLES]

for mask_name in MLFLOW_MASKS:
    paths = [mlflow_dirs[mask_name] / name for name in sample_names if name in mlflow_files[mask_name]]
    all_mask_types.append((mask_name, paths[:N_SAMPLES]))

for mask_type, prefix in QC_MASK_PREFIXES.items():
    paths = [QC_MASKS_PATH / f"{prefix}{name}" for name in sample_names if name in qc_files[mask_type]]
    all_mask_types.append((mask_type, paths[:N_SAMPLES]))

# Plot grid: rows = mask types, cols = samples
n_rows = len(all_mask_types)
n_cols = N_SAMPLES
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
if n_rows == 1:
    axes = [axes]

for row_idx, (mask_type, paths) in enumerate(all_mask_types):
    for col_idx in range(n_cols):
        ax = axes[row_idx][col_idx]
        ax.set_xticks([])
        ax.set_yticks([])
        if col_idx == 0:
            ax.set_ylabel(mask_type, fontsize=12, fontweight="bold")
        if col_idx < len(paths):
            thumb = _read_mask_thumbnail(paths[col_idx])
            if thumb is not None:
                ax.imshow(thumb, cmap="gray")
                ax.set_title(paths[col_idx].name, fontsize=7)
            else:
                ax.text(0.5, 0.5, "read error", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes, color="gray")

fig.suptitle("Sample masks (5 per type)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Tiled dataset validation

For each tiled dataset, download `slides.parquet` and `tiles.parquet` from MLflow, then check:
1. Every slide in `data_mapping.csv` has a row in `slides.parquet` (and vice versa)
2. Every slide in `slides.parquet` has at least one tile in `tiles.parquet`
3. Every `slide_id` in `tiles.parquet` maps to a slide in `slides.parquet`

In [ ]:
# ---- Download tiled datasets from MLflow ----
tiled_data: dict[str, tuple[pd.DataFrame, pd.DataFrame]] = {}

for ds_name, uri in TILED_DATASETS.items():
    local_dir = Path(download_artifacts(uri))
    slides = pd.read_parquet(local_dir / "slides.parquet")
    tiles = pd.read_parquet(local_dir / "tiles.parquet")
    tiled_data[ds_name] = (slides, tiles)
    print(f"{ds_name}: {len(slides)} slides, {len(tiles)} tiles")

In [ ]:
# ---- Validate tiled datasets ----
expected_stems = {Path(p).stem for p in df["path"]}

for ds_name, (slides, tiles) in tiled_data.items():
    print(f"=== {ds_name} ===\n")

    # 1. slides.parquet path stem ↔ data_mapping path stem
    slide_stems = {Path(p).stem for p in slides["path"]}
    missing_from_dataset = expected_stems - slide_stems
    extra_in_dataset = slide_stems - expected_stems

    print(f"Slides in data_mapping: {len(expected_stems)}")
    print(f"Slides in dataset:      {len(slide_stems)}")
    print(f"Missing from dataset:   {len(missing_from_dataset)}")
    print(f"Extra in dataset:       {len(extra_in_dataset)}")

    if missing_from_dataset:
        print(f"\nSlides in data_mapping but NOT in {ds_name}:")
        for p in sorted(missing_from_dataset):
            print(f"  {p}")

    if extra_in_dataset:
        print(f"\nSlides in {ds_name} but NOT in data_mapping:")
        for p in sorted(extra_in_dataset):
            print(f"  {p}")

    # 2. Every slide has at least one tile
    slide_ids = set(slides["id"])
    slide_ids_in_tiles = set(tiles["slide_id"])

    slides_without_tiles = slide_ids - slide_ids_in_tiles
    print(f"\nSlides without any tiles: {len(slides_without_tiles)}")
    if slides_without_tiles:
        for s in sorted(slides_without_tiles):
            print(f"  {s}")

    # 3. No orphan tiles (tile references a slide not in slides.parquet)
    orphan_tile_ids = slide_ids_in_tiles - slide_ids
    print(f"Orphan tile slide_ids:    {len(orphan_tile_ids)}")
    if orphan_tile_ids:
        for sid in sorted(orphan_tile_ids):
            count = int((tiles["slide_id"] == sid).sum())
            print(f"  slide_id={sid}: {count} tiles")

    print()

## Embeddings validation

For each embeddings artifact, check that every slide in `data_mapping.csv` has a corresponding `.parquet` file.

In [ ]:
# ---- Download and validate embeddings ----
# Embedding parquet files are named as "{record_num}.parquet" (e.g. "2023/00835.parquet")
expected_record_nums = set(df["path"])

for emb_name, uri in EMBEDDINGS.items():
    print(f"=== {emb_name} ===\n")
    emb_dir = Path(download_artifacts(uri))
    emb_files = {p.stem for p in emb_dir.iterdir() if p.is_file() and p.suffix == ".parquet"}
    print(f"Embedding files found:  {len(emb_files)}")
    print(f"Slides in data_mapping: {len(expected_record_nums)}")

    missing = expected_record_nums - emb_files
    extra = emb_files - expected_record_nums

    print(f"Missing embeddings:     {len(missing)}")
    if missing:
        for s in sorted(missing):
            print(f"  {s}")

    print(f"Extra embeddings:       {len(extra)}")
    if extra:
        for s in sorted(extra):
            print(f"  {s}")

    print()